In [ ]:
import pandas as pd
import ast
import json

# Load your CSV
df = pd.read_csv('Assessment.csv')

label_colors = {    
    "Acute Assessment": "#FF6666",
    "Reassessment": "#7FFF00",
    "Acute Symptoms": "#FF00FF",
    "Personal Information": "#00FFFF",
    "Personal History": "#8B00FF",
    "Family History": "#FF8000",
    "Drug History": "#00FF80",
    "Theraputic History": "#0040FF",
    "Vegetative History": "#FF0080",
    "Other Social": "#00FFBF",
    "Radiology Examination": "#8000FF",
    "Lab Examination": "#FFBF00",
    "Physical Examination": "#40E0D0",
    "Medications": "#4682B4",
    "Diagnostic Testing": "#DA70D6",
    "Other Treatments": "#FF4500",
    "Follow-up": "#00CED1",
    "Discussion": "#B22222",
    "Referral": "#9932CC"
}

def get_numbered_labeled_parts_with_intent(content, label_json):
    if not isinstance(content, str):
        return []
    if isinstance(label_json, str):
        label_json = label_json.replace(r'\/', '/')
    try:
        spans = ast.literal_eval(label_json)
    except Exception:
        return [content] if content.strip() else []
    spans = [s for s in spans if "start" in s and "end" in s and "labels" in s and s["labels"]]
    spans.sort(key=lambda x: x['start'])
    note_lines = []
    for s in spans:
        start, end, label = s['start'], s['end'], s['labels'][0]
        color = label_colors.get(label, '#000')
        labeled_html = (
            f"<span style='color:{color};' data-toggle='tooltip' data-html='true' title='{label}'>"
            f"{content[start:end]}"
            f"</span>"
        )
        note_lines.append(
            f"<li>{labeled_html} <span style='color:{color}; font-size:14px;'>(<b>{label}</b>)</span></li>"
        )
    return note_lines

# Build legend HTML for the 'Intent' value
legend_html = "".join([
    f"<span style='color:{color}; font-size:15px; margin-right:10px;'>{label}</span>"
    for label, color in label_colors.items()
])

# Replace the note column with the dict structure (as string)
new_notes = []
for _, row in df.iterrows():
    note_lines = get_numbered_labeled_parts_with_intent(row['content'], row['label'])
    numbered_note = f"<ol>{''.join(note_lines)}</ol>"
    # Create the dict as a string
    note_dict = {
        "Intent": legend_html,
        "Note": numbered_note
    }
    new_notes.append(json.dumps(note_dict, ensure_ascii=False))

df['note'] = new_notes

# Save all columns, with the new note column structure
df.to_csv('Assessment_potato_final.csv', index=False)

print("Done! All original columns kept, 'note' column now holds a dict: {'Intent': ..., 'Note': ...}.")
